# Detector de Spam Bilingüe (Inglés + Español)
## Clasificación automática de mensajes con Machine Learning

**Universidad:** Universidad Nacional de Costa Rica  
**Curso:** Inteligencia Artificial  
**Profesor:** _______________________________  
**Estudiante:** _______________________________  
**Fecha:** _______________________________  

---

## Descripción

Este notebook construye un **clasificador automático de spam** capaz de operar tanto en **inglés** como en **español**. Es **auto-contenido**: descarga sus propios datasets (UCI SMS Spam Collection + dataset público en español, con un seed embebido como respaldo) y no depende de ningún archivo externo.

Se implementan y comparan **tres algoritmos** y se aplican técnicas profesionales de validación:

- **Naive Bayes Multinomial** (Bag of Words)
- **Regresión Logística** (TF-IDF con n-gramas de palabras)
- **Regresión Logística** (TF-IDF con n-gramas de caracteres, ideal cross-lingual)
- **Cross-validation k-fold** para detectar overfitting
- **GridSearchCV** para tuning de hiperparámetros
- **Curva de aprendizaje** para diagnosticar si más datos ayudarían
- **Tuning de threshold** para optimizar Recall (métrica prioritaria)
- **Evaluación por idioma** (EN vs ES) para validar capacidad cross-lingual real

---

## Tabla de contenido

1. Objetivos de aprendizaje  
2. Contexto teórico  
3. Análisis PEAS del agente  
4. Importación de librerías  
5. Descarga del dataset inglés (UCI SMS Spam)  
6. Descarga del dataset español público  
7. Carga unificada en DataFrame  
8. Exploración del corpus  
9. Preprocesamiento del texto  
10. Vectorización (BoW, TF-IDF, char n-grams)  
11. División train/test  
12. Entrenamiento de los tres modelos  
13. Cross-validation k-fold (anti-overfitting)  
14. GridSearchCV (tuning de hiperparámetros)  
15. Curva de aprendizaje  
16. Tuning de threshold para Recall  
17. Evaluación por idioma (EN vs ES)  
18. Matriz de confusión y análisis de errores  
19. Predicciones sobre mensajes nuevos  
20. Persistencia del modelo  
21. Conclusiones y limitaciones

# 1. Objetivos de aprendizaje

Al finalizar este notebook, el estudiante será capaz de:

- Explicar el problema de detección de spam como **clasificación binaria supervisada**.
- Descargar y combinar **datasets reales** desde fuentes públicas.
- Aplicar **preprocesamiento de texto** (limpieza, normalización, stopwords).
- Vectorizar texto con **Bag of Words**, **TF-IDF** y **n-gramas de caracteres**.
- Entrenar **Naive Bayes** y **Regresión Logística** con `sklearn.Pipeline`.
- Validar modelos con **cross-validation k-fold** y detectar overfitting.
- Optimizar hiperparámetros con **GridSearchCV**.
- Diagnosticar la capacidad del modelo con **curva de aprendizaje**.
- Ajustar el **threshold de decisión** para maximizar Recall.
- Validar el modelo **por idioma separado** (cross-lingual real).
- Evaluar con métricas profesionales: Accuracy, Precision, Recall, F1, matriz de confusión.
- Persistir el pipeline completo con `joblib` para inferencia futura.

# 2. Contexto teórico

## 2.1 ¿Qué es el spam?

El **spam** son mensajes no solicitados, generalmente con fines comerciales, fraudulentos o de phishing. Detectarlo automáticamente es un problema clásico de **NLP + ML supervisado**.

## 2.2 Pipeline de un clasificador de texto

```text
Texto crudo → Limpieza → Tokenización → Vectorización → Modelo → {spam, ham}
```

## 2.3 Algoritmos usados

### Naive Bayes Multinomial

Aplica el **Teorema de Bayes** asumiendo independencia condicional entre palabras:

$$P(\text{spam} \mid \text{palabras}) = \frac{P(\text{palabras} \mid \text{spam}) \cdot P(\text{spam})}{P(\text{palabras})}$$

Usa **suavizado de Laplace** (α=1.0) para evitar probabilidades cero en palabras nunca vistas.

### Regresión Logística

Aprende pesos $w$ para cada feature y aplica la función sigmoide:

$$P(\text{spam} \mid x) = \frac{1}{1 + e^{-(w^\top x + b)}}$$

## 2.4 Métrica prioritaria: Recall sobre spam

Un **falso negativo** (spam que pasa como ham) es más dañino que un **falso positivo** (ham filtrado como spam). Por eso priorizamos **Recall**:

$$\text{Recall} = \frac{TP}{TP + FN}$$

## 2.5 ¿Qué es cross-lingual?

Un modelo es **cross-lingual** cuando funciona en idiomas para los que tiene poco o ningún dato de entrenamiento. Lo logramos usando **n-gramas de caracteres** (3-5), que capturan patrones sub-palabra (`http`, `gana`, `clic`, `$$$`) que aparecen tanto en spam inglés como español.

# 3. Análisis PEAS del agente

| Componente | Definición |
|------------|-----------|
| **P — Performance** | Accuracy, Precision, Recall, F1. Prioridad: Recall sobre clase `spam` (>90%). |
| **E — Environment** | Mensajes de texto en inglés o español, posiblemente con HTML, URLs, números y errores ortográficos. |
| **A — Actuators** | Etiqueta `spam`/`ham` + nivel de confianza ∈ [0,1] + probabilidades por clase. |
| **S — Sensors** | El texto crudo del mensaje. |

**Tipo de entorno:** totalmente observable, un solo agente, determinístico, episódico, estático.  
**Tipo de agente:** *Model-based* — entrenado una vez sobre el dataset, luego usado en modo inferencia.

# 4. Importación de librerías

In [ ]:
import io
import re
import csv
import html
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    learning_curve,
)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
)

import joblib

warnings.filterwarnings("ignore", category=UserWarning)

try:
    stopwords.words("english")
    stopwords.words("spanish")
except LookupError:
    nltk.download("stopwords", quiet=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías cargadas correctamente.")

# 5. Descarga del dataset inglés — UCI SMS Spam Collection

Bajamos el dataset oficial de UCI: ~5,574 SMS etiquetados como `spam`/`ham`.  
Fuente: <https://archive.ics.uci.edu/dataset/228/sms+spam+collection>

In [ ]:
# Cascada de fuentes para el SMS Spam Collection.
# La URL oficial de UCI a veces tiene certificado SSL expirado; por eso probamos varios mirrors.
SMS_SOURCES = [
    {
        "name": "GitHub mirror (justmarkham)",
        "url": "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv",
        "format": "tsv",
    },
    {
        "name": "UCI oficial",
        "url": "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip",
        "format": "zip",
    },
    {
        "name": "HuggingFace mirror (ucirvine/sms_spam)",
        "url": "https://huggingface.co/datasets/ucirvine/sms_spam/resolve/main/plain_text/train-00000-of-00001.parquet",
        "format": "parquet",
    },
]


def _parse_zip(content):
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        target = [n for n in z.namelist() if "SMSSpamCollection" in n][0]
        with z.open(target) as f:
            return pd.read_csv(
                f, sep="\t", names=["label", "text"],
                quoting=csv.QUOTE_NONE, encoding="utf-8",
                on_bad_lines="skip",
            )


def _parse_parquet(content):
    df = pd.read_parquet(io.BytesIO(content))
    if "sms" in df.columns:
        df = df.rename(columns={"sms": "text"})
    if df["label"].dtype != object:
        df["label"] = df["label"].map({0: "ham", 1: "spam"})
    return df[["label", "text"]]


def _parse_tsv(content):
    return pd.read_csv(
        io.BytesIO(content), sep="\t", names=["label", "text"],
        quoting=csv.QUOTE_NONE, encoding="utf-8",
        on_bad_lines="skip",
    )


_PARSERS = {"zip": _parse_zip, "parquet": _parse_parquet, "tsv": _parse_tsv}


def download_sms_spam():
    """Intenta varias fuentes hasta que una funcione. Tolerante a fallos SSL."""
    last_error = None
    for src in SMS_SOURCES:
        try:
            print(f"  Intentando: {src['name']} ...")
            try:
                resp = requests.get(src["url"], timeout=30)
                resp.raise_for_status()
            except requests.exceptions.SSLError:
                print("    [WARN] SSL fallo, reintentando con verify=False")
                resp = requests.get(src["url"], timeout=30, verify=False)
                resp.raise_for_status()
            df = _PARSERS[src["format"]](resp.content)
            df["label"] = df["label"].astype(str).str.lower()
            df = df[df["label"].isin(["ham", "spam"])].copy()
            df["lang"] = "en"
            print(f"  [OK] {len(df):,} mensajes desde {src['name']}")
            return df[["text", "label", "lang"]].reset_index(drop=True)
        except Exception as exc:
            last_error = exc
            print(f"    [FAIL] {type(exc).__name__}: {str(exc)[:80]}")
    raise RuntimeError(f"Todas las fuentes fallaron. Ultimo error: {last_error}")


print("Descargando SMS Spam Collection ...")
sms_en = download_sms_spam()
print(f"\nDataset ingles cargado: {len(sms_en):,} mensajes")
print(sms_en["label"].value_counts())
sms_en.head()


# 6. Descarga del dataset español público

Para el español usamos **cascada de fuentes** con fallback:

1. Intentar descargar un dataset público desde un mirror de GitHub/HuggingFace.
2. Si falla, usar un **seed curado embebido** dentro del notebook (~150 ejemplos representativos de phishing real en banca, telco, ecommerce y mensajería personal).

Esto garantiza que el notebook funcione incluso sin internet o si las URLs cambian. El seed embebido es **siempre añadido** al dataset descargado para asegurar cobertura mínima de dominios típicos de phishing hispanohablante.

In [ ]:
SPANISH_SEED = [
    ("spam", "FELICIDADES! Has ganado un iPhone 15. Reclama ahora en http://bit.ly/premio-iphone"),
    ("spam", "URGENTE: Tu cuenta bancaria sera suspendida. Verifica tus datos en http://bbva-seguro.co"),
    ("spam", "Has sido seleccionado para un prestamo de 50000 sin requisitos. Llama YA al 800 555 1234"),
    ("spam", "Promo exclusiva!!! 90 por ciento de descuento en Amazon. Compra ahora http://amazon-ofertas.net"),
    ("spam", "GANA DINERO desde casa hasta 5000 semanales. Envia HOLA al 4040"),
    ("spam", "Tu paquete de DHL no pudo ser entregado. Confirma direccion en http://dhl-rastreo.info"),
    ("spam", "Netflix tu suscripcion vencio. Actualiza tu metodo de pago aqui http://netflix-pago.cc"),
    ("spam", "Banco Santander detectamos un acceso sospechoso. Verifica tu identidad en http://santander-acceso.com"),
    ("spam", "Hola guapa tengo fotos para ti. Mira aqui http://chicas-online.xyz"),
    ("spam", "Tu factura de CFE esta vencida. Paga en linea o se cortara la luz en http://cfe-pagos.tk"),
    ("spam", "Loteria Nacional tu numero salio premiado con 100000 EUR. Reclama: premio@loteria-es.org"),
    ("spam", "OFERTA UNICA Viagra y Cialis al 70 por ciento off envio gratis. Pedidos en http://farmacia-rapida.ru"),
    ("spam", "Premio Coca-Cola ganaste un viaje todo pagado a Cancun. Confirma datos en http://cocacola-promo.com"),
    ("spam", "Reactiva tu cuenta de WhatsApp antes de 24h o sera eliminada en http://whatsapp-verifica.net"),
    ("spam", "Aprovecha Bitcoin gratis cada hora registrate ya en http://btc-free-faucet.io"),
    ("spam", "Tu tarjeta Visa fue bloqueada por seguridad. Desbloquea en http://visa-soporte.cc"),
    ("spam", "GRATIS Curso de ingles avanzado en 7 dias. Cupos limitados en http://aprende-rapido.online"),
    ("spam", "Has recibido una transferencia de 1250 USD. Reclama tu codigo aqui http://paypal-confirma.org"),
    ("spam", "SAT tienes una devolucion pendiente de 8400. Ingresa CURP en http://sat-devoluciones.mx-info.com"),
    ("spam", "Sorteo Mercado Libre ganaste 25000 MXN en compras. Reclama en http://meli-sorteo.win"),
    ("spam", "Empleo desde casa 1500 al dia. Sin experiencia. WhatsApp +52 55 1234 5678"),
    ("spam", "Tu cuenta de Instagram sera eliminada por copyright. Apela aqui en http://insta-soporte.help"),
    ("spam", "Inversion garantizada 30 por ciento mensual en cripto. Empieza con solo 100 en http://cripto-rico.net"),
    ("spam", "FELICITACIONES Eres el visitante 1000000. Reclama tu MacBook gratis en http://promo-apple.fake"),
    ("spam", "Tu Spotify premium expira hoy. Renueva con un click en http://spotify-renovar.cc"),
    ("spam", "BBVA Mexico nuevo intento de acceso desde Rusia. Bloquea aqui http://bbva-seguridad.tk"),
    ("spam", "Aumenta el tamano de tu pene 5cm en 2 semanas garantizado. Pedidos en http://pastillas-hombres.shop"),
    ("spam", "Tu pedido de Aliexpress requiere pago de aduana de 4 USD. Paga aqui http://aduana-mx.online"),
    ("spam", "OFERTON Liverpool pantalla 65 pulgadas a 999. Solo hoy en http://liverpool-mx.deals"),
    ("spam", "Has ganado el sorteo del Banco BBVA. Para reclamar envia tus datos a premios@bbva-mx.cc"),
    ("spam", "Hola amor soy Maria. Conocenos en http://citas-locales.cc"),
    ("spam", "Tu Telcel tiene 5GB gratis. Activa marcando estrella 2025 o entra a http://telcel-promo.mx"),
    ("spam", "ALERTA malware detectado en tu PC. Limpia gratis con http://antivirus-free.dl"),
    ("spam", "Casino Royal bono de bienvenida 500 sin deposito. Juega ya en http://casino-royal.bet"),
    ("spam", "Reclama tu reembolso de Hacienda de 432 EUR antes del viernes en http://agencia-tributaria.es-info.com"),
    ("spam", "Mercado Pago tu cuenta sera limitada. Verifica en http://mercado-pago-verifica.cc"),
    ("spam", "Trabaja desde tu celular ganando 500 USD diarios. Info al WhatsApp +1 305 555 0199"),
    ("spam", "Solo por hoy iPhone 14 Pro a 199 EUR. Stock limitado en http://apple-ofertas-eu.shop"),
    ("spam", "Tu DNI electronico expiro. Renueva en linea en http://dni-renovar.gob-info.es"),
    ("spam", "Has sido elegido para participar en estudio remunerado. 200 EUR por encuesta http://encuestas-pagadas.cc"),
    ("spam", "GANA 1000 EUROS AHORA Responde este SMS con la palabra GANAR"),
    ("spam", "Apple Pay tu cuenta requiere verificacion urgente. Confirma en http://apple-id-verify.org"),
    ("spam", "Banco Galicia detectamos compras sospechosas. Cancela en http://galicia-seguro.com.ar"),
    ("spam", "Tu factura de Movistar esta vencida. Paga aqui o se suspendera tu linea http://movistar-pago.tk"),
    ("spam", "Hola te envio un video privado. Abrelo aqui http://contenido-adulto.xxx"),
    ("spam", "Ultima oportunidad 80 por ciento off en zapatos Nike y Adidas. Compra en http://outlet-deportivo.cc"),
    ("spam", "Tu cuenta de Steam fue suspendida. Recupera tu acceso en http://steam-recovery.fake"),
    ("spam", "Felicidades Has ganado un cupon de 500 en Walmart. Reclama en http://walmart-cupones.com"),
    ("spam", "OFERTA IRRESISTIBLE cripto trading curso gratis. Aprende en http://forex-millonario.tk"),
    ("spam", "Tu Uber tiene un descuento del 50 por ciento en tu proximo viaje. Activa en http://uber-promo.fake"),
    ("spam", "Banco Bancomer tu app esta desactualizada. Actualiza ahora en http://bancomer-app.tk"),
    ("spam", "Tu suscripcion a YouTube Premium expiro. Renueva en http://youtube-renovar.cc"),
    ("spam", "GANASTE Un viaje a Disney. Confirma asistencia en http://disney-sorteo.fake"),
    ("spam", "Aviso tu cuenta Google sera cerrada en 24h. Verifica en http://google-acceso.tk"),
    ("spam", "OFERTA EXCLUSIVA laptops HP a mitad de precio. Compra en http://hp-ofertas.cc"),
    ("spam", "Tu pedido Amazon esta retenido en aduana. Paga 2 USD en http://amazon-aduana.fake"),
    ("spam", "Banco Itau tu token expiro. Genera uno nuevo en http://itau-token.com.br"),
    ("spam", "Ofertas de empleo desde casa pagando 800 USD por semana. Aplica en http://trabajo-online.tk"),
    ("spam", "Tu paquete de Correos esta retenido. Paga tasa aduanera en http://correos-pago.es"),
    ("spam", "Premio Loto ganaste 5000000 MXN. Datos en http://loto-mx.fake"),
    ("spam", "Tu numero salio seleccionado en sorteo Telmex. Reclama en http://telmex-sorteo.cc"),
    ("spam", "BUYS NOW descuento del 99 por ciento en Rolex replicas. Pedidos en http://rolex-baratos.cn"),
    ("spam", "Tu cuenta bancaria HSBC fue bloqueada. Reactiva en http://hsbc-bloqueo.fake"),
    ("spam", "Conoce mujeres solteras en tu zona ahora mismo. Registrate en http://citas-rapidas.cc"),
    ("spam", "PROMO BIMBO gana un viaje a Cancun comprando 3 paquetes. Codigo en http://bimbo-promo.tk"),
    ("spam", "Aviso fiscal regularize su situacion antes del 31. Detalles en http://sat-aviso.cc"),
    ("spam", "Has ganado un Tesla Model 3. Para reclamar envia tu DNI a premios@tesla-promo.fake"),
    ("spam", "OFERTA RELAMPAGO hoy iPhone 16 a 99 EUR. Aprovecha en http://flash-sale.tk"),
    ("spam", "BCP confirma tu transferencia de 5000 soles en http://bcp-confirmacion.pe"),
    ("spam", "Tu cuenta de Facebook sera deshabilitada. Apela aqui http://facebook-soporte.tk"),
    ("spam", "Gratis ringtones fondos de pantalla y juegos. Descarga en http://moviles-gratis.cc"),
    ("spam", "DHL paga 3 EUR de aduana o pierdes tu paquete en http://dhl-pago.fake"),
    ("spam", "PROMO gana 1000 puntos Smart-Fit gratis. Activa en http://smartfit-promo.cc"),
    ("spam", "Aviso Banamex tu chequera expirara pronto. Renueva en http://banamex-renovar.tk"),
    ("spam", "Has sido pre-aprobado para tarjeta Platinum sin verificacion. Solicita en http://tarjeta-platinum.cc"),
    ("spam", "PARTIDOS HOY pronosticos premium gratis por 24h en http://apuestas-vip.tk"),
    ("spam", "Tu cuenta PayPal tiene un pago retenido de 230 USD. Reclama en http://paypal-pago.fake"),
    ("spam", "FELICIDADES Has sido elegido para probar productos Apple gratis http://apple-tester.cc"),
    ("spam", "Curso GRATIS de marketing digital con certificado. Inscripcion en http://marketing-gratis.tk"),
    ("ham", "Hola como estas Te paso a buscar a las 7 de la noche para ir al cine"),
    ("ham", "Recordatorio junta de equipo manana a las 10am en la sala 3"),
    ("ham", "Mama dice que llegues temprano porque tenemos cena familiar"),
    ("ham", "Te mando el documento revisado dime si necesitas cambios antes del viernes"),
    ("ham", "Gracias por la ayuda con el codigo ya logre que compile"),
    ("ham", "Hoy salgo tarde de la oficina no me esperen para cenar"),
    ("ham", "Pasame la receta del flan que hiciste el domingo estaba delicioso"),
    ("ham", "Confirmo asistencia a la reunion del jueves llevo el reporte impreso"),
    ("ham", "El nino tiene fiebre voy a llevarlo al medico esta tarde"),
    ("ham", "Quedamos en el bar de siempre a las 9 Avisame si cambias de plan"),
    ("ham", "Buenos dias le envio el presupuesto para la remodelacion segun lo platicado"),
    ("ham", "Acabo de llegar a la oficina en 10 minutos te marco para revisar el contrato"),
    ("ham", "Feliz cumpleanos hermanito que cumplas muchos mas Un abrazo enorme"),
    ("ham", "El profesor cambio la fecha del examen para el martes 15"),
    ("ham", "Estoy en el supermercado quieres que te lleve algo"),
    ("ham", "El partido empieza a las 8 vienes a verlo a la casa"),
    ("ham", "Ya envie el correo al cliente con la cotizacion actualizada"),
    ("ham", "Pase por farmacia las pastillas que pediste no las tenian"),
    ("ham", "El vuelo aterriza a las 6 45am tomo un Uber al hotel"),
    ("ham", "Que tal el viaje Manda fotos cuando puedas"),
    ("ham", "Necesito que me cubras la guardia del sabado te debo una"),
    ("ham", "Encontre el libro que querias en la biblioteca te lo presto el lunes"),
    ("ham", "El plomero viene manana entre 9 y 11 puedes recibirlo"),
    ("ham", "Adjunto las minutas de la reunion del miercoles para tu revision"),
    ("ham", "Nos juntamos en mi casa para estudiar el examen de calculo"),
    ("ham", "Ya pague la luz te paso comprobante por whatsapp"),
    ("ham", "El doctor dijo que esta todo bien solo descanso por una semana"),
    ("ham", "Me encanto la pelicula gracias por recomendarmela"),
    ("ham", "Voy a estar fuera de la oficina del 5 al 12 cualquier urgencia escribe a Carlos"),
    ("ham", "Acuerdate de comprar pan cuando regreses ya casi no queda"),
    ("ham", "Profe podriamos revisar la tarea 4 en la asesoria de manana"),
    ("ham", "El tren va con 20 minutos de retraso llegare un poco tarde"),
    ("ham", "Ya esta listo el reporte trimestral lo subi al drive compartido"),
    ("ham", "Tu paquete llego a la oficina lo dejo en tu escritorio"),
    ("ham", "Felicidades por el ascenso te lo merecias muchisimo"),
    ("ham", "Tenemos la mesa reservada para las 8 30 a nombre de Lopez"),
    ("ham", "El abuelo pregunto por ti llamale cuando puedas"),
    ("ham", "Guardame un pedazo de pastel que voy a llegar tarde"),
    ("ham", "El servidor estuvo caido pero ya esta funcionando otra vez"),
    ("ham", "Ya termine el primer capitulo de la tesis me dices que opinas"),
    ("ham", "Si quieres puedo pasar por ti a la salida del trabajo"),
    ("ham", "Compre los boletos para el concierto son fila 12 centro"),
    ("ham", "Hola tia gracias por el regalo los chicos estan felices"),
    ("ham", "El plomero arreglo la fuga cobro 800 pesos"),
    ("ham", "Cuando podemos agendar la presentacion con el cliente"),
    ("ham", "Ya me dieron los resultados del laboratorio todo en rangos normales"),
    ("ham", "Acuerdate que el lunes es feriado la oficina cierra"),
    ("ham", "Te dejo las llaves debajo de la maceta como siempre"),
    ("ham", "Voy saliendo de casa en 30 min llego al restaurante"),
    ("ham", "Me presto el coche tu hermano para llevar las cosas"),
    ("ham", "Termine el ejercicio que me mandaste te lo envio mas tarde por correo"),
    ("ham", "La impresora se quedo sin tinta podrias traer un cartucho"),
    ("ham", "Nos vemos directo en el aeropuerto mi avion llega a las 3"),
    ("ham", "Manana hay clase a las 8 o cambiaron horario"),
    ("ham", "Te aviso cuando tenga el calendario definitivo de vacaciones"),
    ("ham", "Saliendo de la oficina en 20 minutos paso a buscarte"),
    ("ham", "La presentacion del cliente quedo agendada para el viernes 10am"),
    ("ham", "Hoy en la noche cenamos pizza paso por una de camino"),
    ("ham", "Llame a la aseguradora mandan a un perito el lunes"),
    ("ham", "Me cobraron mal el recibo de la luz hay que reclamar"),
    ("ham", "Hijo recuerda llevar sueter va a bajar la temperatura"),
    ("ham", "Quieres que te recoja del aeropuerto avisame la hora del vuelo"),
    ("ham", "El pediatra dio la cita para el jueves a las 4"),
    ("ham", "Necesito revisar contigo el contrato antes de firmarlo"),
    ("ham", "Mande el pago del internet ya quedo al corriente"),
    ("ham", "El equipo gano el partido por 3 a 1 fue una buena noche"),
    ("ham", "Pedimos sushi para la cena hay un descuento en Rappi"),
    ("ham", "El cliente acepto la propuesta firmamos manana"),
    ("ham", "Que bueno verte ayer hay que repetir pronto"),
    ("ham", "Ya recogi a los ninos del cole vamos a casa"),
    ("ham", "Pasame la foto del recibo que me la pidio contabilidad"),
    ("ham", "Quedamos confirmados para el sabado llevo el postre"),
    ("ham", "Manana tengo cita con el dentista voy a llegar mas tarde"),
    ("ham", "El nuevo proyecto arranca el lunes ya nos asignaron equipo"),
    ("ham", "Acuerdate de descongelar la carne para la cena"),
    ("ham", "El curso de Python que comparti es bueno te lo recomiendo"),
    ("ham", "Cancelaron la reunion de las 3 agendaron para el martes"),
    ("ham", "Voy llegando a la casa abre por favor"),
    ("ham", "Recuerda que el examen final es el viernes a las 9am"),
    ("ham", "Ya pague la inscripcion del gimnasio empezamos manana"),
]


def try_download_spanish_spam():
    """Intenta descargar un dataset publico en espanol; devuelve None si todas fallan."""
    candidate_urls = [
        "https://raw.githubusercontent.com/UrielUriel/spam-dataset-spanish/main/spam_spanish.csv",
        "https://huggingface.co/datasets/MarcOrfilaCarreras/spanish-spam-sms/resolve/main/spam_es.csv",
    ]
    for url in candidate_urls:
        try:
            print(f"  Intentando: {url}")
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            df = pd.read_csv(io.StringIO(resp.text))
            cols = {c.lower(): c for c in df.columns}
            label_col = cols.get("label") or cols.get("class") or cols.get("category")
            text_col = cols.get("text") or cols.get("message") or cols.get("sms")
            if not label_col or not text_col:
                raise ValueError(f"Columnas no encontradas: {df.columns.tolist()}")
            df = df[[label_col, text_col]].rename(columns={label_col: "label", text_col: "text"})
            df["label"] = df["label"].astype(str).str.lower()
            df = df[df["label"].isin(["ham", "spam"])].copy()
            if len(df) < 20:
                raise ValueError(f"Dataset muy pequeno ({len(df)} filas)")
            print(f"  [OK] Descargado de: {url}")
            df["lang"] = "es"
            return df[["text", "label", "lang"]].reset_index(drop=True)
        except Exception as exc:
            print(f"  [FAIL] {type(exc).__name__}: {str(exc)[:80]}")
    print("  [INFO] Ninguna URL publica respondio; usando solo el seed embebido.")
    return None


print("Buscando dataset publico en espanol ...")
downloaded_es = try_download_spanish_spam()

seed_df = pd.DataFrame([{"text": t, "label": l, "lang": "es"} for l, t in SPANISH_SEED])

if downloaded_es is not None and len(downloaded_es) > 0:
    sms_es = pd.concat([downloaded_es, seed_df], ignore_index=True)
    print(f"\nDataset espanol = descargado ({len(downloaded_es)}) + seed ({len(seed_df)}) = {len(sms_es)}")
else:
    sms_es = seed_df
    print(f"\nDataset espanol = solo seed embebido = {len(sms_es)}")

print(sms_es["label"].value_counts())
sms_es.head()

# 7. Carga unificada en DataFrame

Unimos ambos datasets en un solo DataFrame con columnas `text`, `label`, `lang`. Eliminamos duplicados para evitar inflar el modelo con datos repetidos.

In [ ]:
df = pd.concat([sms_en, sms_es], ignore_index=True)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
df = df[df["text"].notna() & (df["text"].str.strip().str.len() > 0)].reset_index(drop=True)

print(f"Total de mensajes (sin duplicados): {len(df):,}")
df.sample(5, random_state=RANDOM_STATE)

# 8. Exploración del corpus

In [ ]:
distribution = df.groupby(["lang", "label"]).size().unstack(fill_value=0)
print("Distribución de clases por idioma:")
print(distribution)

spam_ratio = (df["label"] == "spam").mean()
print(f"\nProporción global de spam: {spam_ratio:.2%}")
print(f"Proporción global de ham:  {1 - spam_ratio:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["label"].value_counts().plot(kind="bar", ax=axes[0], color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Distribución global de clases")
axes[0].set_ylabel("Cantidad de mensajes")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
distribution.plot(kind="bar", stacked=True, ax=axes[1], color=["#2ecc71", "#e74c3c"])
axes[1].set_title("Distribución por idioma")
axes[1].set_ylabel("Cantidad de mensajes")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df["length"] = df["text"].str.len()
print("Longitud promedio de mensajes (caracteres) por clase:")
print(df.groupby("label")["length"].describe()[["mean", "50%", "min", "max"]])

fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [("ham", "#2ecc71"), ("spam", "#e74c3c")]:
    df.loc[df["label"] == label, "length"].clip(upper=500).hist(
        bins=40, alpha=0.6, label=label, color=color, ax=ax
    )
ax.set_xlabel("Longitud (caracteres, recortado a 500)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de longitud por clase")
ax.legend()
plt.show()

### Observaciones sobre el corpus

- El dataset tiene **desbalance** (más ham que spam). Esto justifica el uso de `class_weight="balanced"` en Regresión Logística.
- El **español está sub-representado** frente al inglés. Por eso introduciremos también un vectorizador de **caracteres** que transfiere mejor entre idiomas.

# 9. Preprocesamiento del texto

Limpieza paso a paso:

1. Decodificar entidades HTML.
2. Eliminar etiquetas HTML.
3. Reemplazar URLs, emails y números por **tokens especiales** (preservan la señal sin sobreajustar al string exacto).
4. Pasar a minúsculas.
5. Eliminar caracteres no alfabéticos (preservando tildes y `ñ`).
6. Normalizar espacios en blanco.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+")
NUMBER_RE = re.compile(r"\b\d+(?:[.,]\d+)*\b")
HTML_TAG_RE = re.compile(r"<[^>]+>")
NON_WORD_RE = re.compile(r"[^\wáéíóúñü\s-]", flags=re.IGNORECASE)
WHITESPACE_RE = re.compile(r"\s+")

STOPWORDS = set(stopwords.words("english")) | set(stopwords.words("spanish"))
print(f"Stopwords combinadas (EN + ES): {len(STOPWORDS):,}")


def clean_text(text):
    """Normaliza un texto crudo aplicando todas las reglas de limpieza."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = HTML_TAG_RE.sub(" ", text)
    text = URL_RE.sub(" __url__ ", text)
    text = EMAIL_RE.sub(" __email__ ", text)
    text = NUMBER_RE.sub(" __num__ ", text)
    text = text.lower()
    text = NON_WORD_RE.sub(" ", text)
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text


ejemplos = df.sample(3, random_state=RANDOM_STATE)
for _, row in ejemplos.iterrows():
    print(f"[{row['label'].upper():4} {row['lang'].upper()}] Original: {row['text'][:100]!r}")
    print(f"              Limpio:   {clean_text(row['text'])[:100]!r}")
    print()

In [ ]:
df["text_clean"] = df["text"].apply(clean_text)
df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)
print(f"Mensajes válidos tras limpieza: {len(df):,}")

# 10. Vectorización (BoW, TF-IDF, char n-grams)

El modelo necesita números, no texto. Usaremos tres estrategias:

| Estrategia | Fórmula | Ventaja |
|-----------|---------|---------|
| **Bag of Words** | conteo de cada palabra | rápido, interpretable |
| **TF-IDF (palabras)** | $tf \cdot \log(N/df)$ | penaliza palabras muy comunes |
| **TF-IDF char_wb** | n-gramas de caracteres (3,5) | robusto cross-lingual y a typos |

Cada vectorizador se aplica **dentro de un `Pipeline`** en la siguiente sección, para evitar data leakage durante la validación cruzada.

# 11. División train/test

- **80% entrenamiento / 20% prueba**
- **`stratify=label`** preserva la proporción spam/ham en ambos splits.
- **`random_state=42`** para reproducibilidad.
- Conservamos `lang` para evaluar después por idioma.

In [ ]:
X = df["text_clean"].tolist()
y = df["label"].tolist()
lang_arr = df["lang"].tolist()

X_train, X_test, y_train, y_test, lang_train, lang_test = train_test_split(
    X, y, lang_arr,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {len(X_train):,} mensajes")
print(f"Test:  {len(X_test):,} mensajes")
print(f"\nProporción spam en train: {(np.array(y_train) == 'spam').mean():.2%}")
print(f"Proporción spam en test:  {(np.array(y_test) == 'spam').mean():.2%}")
print(f"\nIdiomas en test:")
print(pd.Series(lang_test).value_counts())

## Helper de evaluación

In [ ]:
def evaluate_model(name, model, X_test, y_test, verbose=True):
    """Evalúa un modelo y devuelve dict con accuracy, precision, recall, f1."""
    y_pred = model.predict(X_test)
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label="spam", zero_division=0),
        "recall": recall_score(y_test, y_pred, pos_label="spam", zero_division=0),
        "f1": f1_score(y_test, y_pred, pos_label="spam", zero_division=0),
    }
    if verbose:
        print(f"\n{'=' * 60}\n  {name}\n{'=' * 60}")
        print(f"Accuracy:  {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}  (sobre clase 'spam')")
        print(f"Recall:    {metrics['recall']:.4f}  (sobre clase 'spam')")
        print(f"F1-Score:  {metrics['f1']:.4f}")
        print("\nReporte detallado:")
        print(classification_report(y_test, y_pred, zero_division=0))
    return metrics, y_pred

# 12. Entrenamiento de los tres modelos

Cada modelo es un `Pipeline` end-to-end (vectorizador + clasificador), de modo que el objeto serializado contiene **todo** lo necesario para inferencia.

In [ ]:
# Modelo 1 — Naive Bayes Multinomial (BoW)
nb_pipeline = Pipeline([
    ("vec", CountVectorizer(
        ngram_range=(1, 2), min_df=2, max_df=0.95,
        stop_words=list(STOPWORDS),
    )),
    ("clf", MultinomialNB(alpha=1.0)),
])
nb_pipeline.fit(X_train, y_train)
nb_metrics, nb_pred = evaluate_model("Naive Bayes (BoW)", nb_pipeline, X_test, y_test)

In [ ]:
# Modelo 2 — Regresión Logística + TF-IDF (palabras)
lr_pipeline = Pipeline([
    ("vec", TfidfVectorizer(
        ngram_range=(1, 2), min_df=2, max_df=0.95,
        sublinear_tf=True, stop_words=list(STOPWORDS),
    )),
    ("clf", LogisticRegression(
        C=1.0, max_iter=1000, solver="liblinear",
        class_weight="balanced",
    )),
])
lr_pipeline.fit(X_train, y_train)
lr_metrics, lr_pred = evaluate_model(
    "Logistic Regression (TF-IDF palabras)", lr_pipeline, X_test, y_test
)

In [ ]:
# Modelo 3 — Regresión Logística + TF-IDF char_wb (cross-lingual)
lr_char_pipeline = Pipeline([
    ("vec", TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5),
        min_df=2, max_df=0.95, sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        C=1.0, max_iter=1000, solver="liblinear",
        class_weight="balanced",
    )),
])
lr_char_pipeline.fit(X_train, y_train)
lr_char_metrics, lr_char_pred = evaluate_model(
    "Logistic Regression (char n-grams)", lr_char_pipeline, X_test, y_test
)

In [ ]:
comparison = pd.DataFrame([nb_metrics, lr_metrics, lr_char_metrics]).set_index("model")
fig, ax = plt.subplots(figsize=(10, 5))
comparison.plot(kind="bar", ax=ax, colormap="viridis")
ax.set_title("Comparación de métricas por modelo (test set)")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
comparison.round(4)

# 13. Cross-validation k-fold (anti-overfitting)

## ¿Por qué hacer esto?

Un único split 80/20 puede dar métricas engañosas: tal vez por casualidad el 20% de test era muy fácil. La **validación cruzada k-fold** divide el train en 5 trozos y entrena/evalúa 5 veces, dando un estimado más confiable de cómo generaliza el modelo.

**Detección de overfitting:** si `train_score - test_score > 0.05`, el modelo memorizó el train y no generaliza.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

cv_results = {}
for name, pipeline in [
    ("Naive Bayes", nb_pipeline),
    ("Logistic (TF-IDF)", lr_pipeline),
    ("Logistic (char)", lr_char_pipeline),
]:
    print(f"Cross-validation ({name}) ...")
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring,
        return_train_score=True, n_jobs=-1,
    )
    cv_results[name] = scores

summary = []
for name, scores in cv_results.items():
    for metric in scoring:
        train_mean = scores[f"train_{metric}"].mean()
        test_mean = scores[f"test_{metric}"].mean()
        test_std = scores[f"test_{metric}"].std()
        summary.append({
            "model": name, "metric": metric,
            "train": round(train_mean, 4),
            "cv_mean": round(test_mean, 4),
            "cv_std": round(test_std, 4),
            "gap": round(train_mean - test_mean, 4),
        })

cv_summary = pd.DataFrame(summary)
cv_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, scores) in zip(axes, cv_results.items()):
    f1_train = scores["train_f1_macro"]
    f1_test = scores["test_f1_macro"]
    folds = range(1, len(f1_train) + 1)
    ax.plot(folds, f1_train, marker="o", label="Train", color="#3498db")
    ax.plot(folds, f1_test, marker="s", label="Validación (CV)", color="#e67e22")
    ax.set_title(name)
    ax.set_xlabel("Fold")
    ax.set_ylim(0.7, 1.02)
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("F1 macro")
plt.suptitle("Train vs Validación por fold — ¿hay overfitting?", y=1.02)
plt.tight_layout()
plt.show()

print("\nDiagnóstico:")
for name, scores in cv_results.items():
    gap = scores["train_f1_macro"].mean() - scores["test_f1_macro"].mean()
    status = "OK" if gap < 0.05 else ("ALERTA" if gap < 0.10 else "OVERFITTING")
    print(f"  {name:20s}  gap F1 = {gap:+.4f}  ->  [{status}]")

### Lectura del resultado

- Si **train ≈ CV**: el modelo generaliza bien, no hay overfitting.
- Si **train >> CV**: el modelo está memorizando; aplicaríamos más regularización o `min_df` mayor.
- La **desviación estándar entre folds** indica estabilidad — un std bajo significa que el modelo es consistente sin importar qué subconjunto vea.

# 14. GridSearchCV — tuning de hiperparámetros

Hasta ahora usamos hiperparámetros fijos (`C=1.0`, `ngram_range=(1,2)`, etc.). `GridSearchCV` prueba **todas las combinaciones** de un grid y elige la mejor por validación cruzada.

Limitamos el grid a 18 combinaciones para que ejecute rápido en CPU.

In [ ]:
param_grid = {
    "vec__ngram_range": [(1, 1), (1, 2)],
    "vec__min_df": [1, 2, 5],
    "clf__C": [0.1, 1.0, 10.0],
}

grid = GridSearchCV(
    lr_pipeline, param_grid=param_grid,
    cv=3, scoring="f1_macro", n_jobs=-1, verbose=1,
)

print("Ejecutando GridSearchCV (18 combinaciones x 3 folds) ...")
grid.fit(X_train, y_train)

print(f"\nMejor F1 macro (CV): {grid.best_score_:.4f}")
print("Mejores hiperparámetros:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")

best_pipeline = grid.best_estimator_
best_metrics, best_pred = evaluate_model(
    "Logistic tuneado (GridSearch)", best_pipeline, X_test, y_test
)

In [ ]:
improvement = pd.DataFrame([lr_metrics, best_metrics]).set_index("model").round(4)
improvement

# 15. Curva de aprendizaje

¿Conseguir **más datos** mejoraría al modelo? La curva de aprendizaje lo dice:

- Si **train y validación convergen alto** → tenemos suficiente data.
- Si hay una **brecha grande** entre ambas → más datos ayudarían.
- Si **ambas curvas están bajas y planas** → el modelo es demasiado simple (underfitting).

In [ ]:
print("Generando curva de aprendizaje (puede tardar ~1-2 min) ...")
train_sizes, train_scores, val_scores = learning_curve(
    best_pipeline, X_train, y_train,
    cv=3, scoring="f1_macro",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1, random_state=RANDOM_STATE,
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, train_mean, "o-", color="#3498db", label="Train F1")
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.2, color="#3498db")
ax.plot(train_sizes, val_mean, "s-", color="#e67e22", label="Validación F1")
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.2, color="#e67e22")
ax.set_xlabel("Tamaño del set de entrenamiento")
ax.set_ylabel("F1 macro")
ax.set_title("Curva de aprendizaje — ¿más datos ayudarían?")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nF1 final (train):       {train_mean[-1]:.4f}")
print(f"F1 final (validación):  {val_mean[-1]:.4f}")
print(f"Brecha:                 {train_mean[-1] - val_mean[-1]:+.4f}")

# 16. Tuning de threshold para Recall

## ¿Por qué?

Por defecto, `predict()` usa **threshold = 0.5** sobre `predict_proba`. Pero en spam queremos **maximizar Recall**: aunque generemos algunos falsos positivos, no queremos dejar pasar spam.

**Estrategia:** buscar el threshold más bajo posible que mantenga Precision ≥ 0.85, y reportar las métricas en ambos thresholds.

In [ ]:
classes = best_pipeline.classes_
spam_idx = list(classes).index("spam")
y_proba = best_pipeline.predict_proba(X_test)[:, spam_idx]
y_test_bin = np.array([1 if v == "spam" else 0 for v in y_test])

precision_vals, recall_vals, thresholds = precision_recall_curve(y_test_bin, y_proba)

PRECISION_FLOOR = 0.85
mask = precision_vals[:-1] >= PRECISION_FLOOR
if mask.any():
    valid_recalls = recall_vals[:-1][mask]
    valid_thresholds = thresholds[mask]
    best_idx = np.argmax(valid_recalls)
    optimal_threshold = float(valid_thresholds[best_idx])
else:
    optimal_threshold = 0.5
    print(f"  [WARN] Ningún threshold mantiene Precision >= {PRECISION_FLOOR}")

print(f"Threshold óptimo (max Recall con P >= {PRECISION_FLOOR}): {optimal_threshold:.4f}")

y_pred_optimal = (y_proba >= optimal_threshold).astype(int)
opt_p = precision_score(y_test_bin, y_pred_optimal, zero_division=0)
opt_r = recall_score(y_test_bin, y_pred_optimal, zero_division=0)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall_vals, precision_vals, color="#3498db", lw=2)
ax.axhline(PRECISION_FLOOR, color="#95a5a6", linestyle="--", alpha=0.5,
           label=f"Precision = {PRECISION_FLOOR}")
ax.scatter([opt_r], [opt_p], color="#e74c3c", s=120, zorder=5,
           label=f"Óptimo (t={optimal_threshold:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Curva Precision-Recall — clase 'spam'")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def metrics_at_threshold(y_proba, y_true_bin, threshold):
    y_pred_bin = (y_proba >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision": precision_score(y_true_bin, y_pred_bin, zero_division=0),
        "recall": recall_score(y_true_bin, y_pred_bin, zero_division=0),
        "f1": f1_score(y_true_bin, y_pred_bin, zero_division=0),
    }

thr_comparison = pd.DataFrame([
    metrics_at_threshold(y_proba, y_test_bin, 0.5),
    metrics_at_threshold(y_proba, y_test_bin, optimal_threshold),
], index=["default (0.5)", f"óptimo ({optimal_threshold:.3f})"])
thr_comparison.round(4)

### Interpretación

Bajar el threshold (ej. 0.5 → 0.3) hace que el modelo sea **más sensible** a spam: clasifica más mensajes como spam, lo que **sube Recall a costa de Precision**. Es la decisión correcta cuando el costo de un falso negativo es mayor que el de un falso positivo.

# 17. Evaluación por idioma (cross-lingual real)

El F1 global puede ocultar problemas: el modelo podría tener F1=0.95 global pero F1=0.50 en español. Aquí lo verificamos.

In [ ]:
df_test_eval = pd.DataFrame({
    "text": X_test, "label": y_test,
    "lang": lang_test, "pred": best_pred,
})

per_lang_metrics = []
for current_lang in sorted(df_test_eval["lang"].unique()):
    subset = df_test_eval[df_test_eval["lang"] == current_lang]
    if len(subset) < 5:
        print(f"[{current_lang}] muestra muy pequeña ({len(subset)}), omitido.")
        continue
    per_lang_metrics.append({
        "lang": current_lang,
        "n_mensajes": len(subset),
        "accuracy": accuracy_score(subset["label"], subset["pred"]),
        "precision": precision_score(subset["label"], subset["pred"], pos_label="spam", zero_division=0),
        "recall": recall_score(subset["label"], subset["pred"], pos_label="spam", zero_division=0),
        "f1": f1_score(subset["label"], subset["pred"], pos_label="spam", zero_division=0),
    })
    print(f"\n=== Reporte detallado [{current_lang.upper()}] ({len(subset)} mensajes) ===")
    print(classification_report(subset["label"], subset["pred"], zero_division=0))

per_lang_df = pd.DataFrame(per_lang_metrics).set_index("lang")
per_lang_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
per_lang_df[["accuracy", "precision", "recall", "f1"]].plot(
    kind="bar", ax=ax, colormap="viridis"
)
ax.set_title("Métricas por idioma — ¿el modelo es realmente cross-lingual?")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Conclusión cross-lingual

- **EN F1** suele estar por encima de 0.90 (es el idioma dominante en train).
- **ES F1** depende del tamaño del seed/dataset español disponible. Si está cerca de 0.70+, el modelo **sí transfiere** entre idiomas; si está por debajo de 0.50, necesitamos más datos en español o un modelo multilingüe (BERT).

# 18. Matriz de confusión y análisis de errores

In [ ]:
cm = confusion_matrix(y_test, best_pred, labels=["ham", "spam"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["ham", "spam"])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Matriz de confusión — modelo tuneado")
plt.show()

tn, fp, fn_count, tp = cm.ravel()
print(f"Verdaderos negativos (ham -> ham):   {tn}")
print(f"Falsos positivos    (ham -> spam):  {fp}  <- ham bloqueado por error")
print(f"Falsos negativos    (spam -> ham):  {fn_count}  <- spam que pasó el filtro")
print(f"Verdaderos positivos(spam -> spam): {tp}")

In [ ]:
errors = df_test_eval[df_test_eval["label"] != df_test_eval["pred"]]
print(f"Total de errores: {len(errors)} de {len(df_test_eval)} ({len(errors)/len(df_test_eval):.2%})")

print("\n--- Falsos negativos (spam que pasó como ham) ---")
for _, row in errors[(errors["label"] == "spam") & (errors["pred"] == "ham")].head(3).iterrows():
    print(f"  [{row['lang']}] {row['text'][:150]}")

print("\n--- Falsos positivos (ham marcado como spam) ---")
for _, row in errors[(errors["label"] == "ham") & (errors["pred"] == "spam")].head(3).iterrows():
    print(f"  [{row['lang']}] {row['text'][:150]}")

# 19. Predicciones sobre mensajes nuevos (EN + ES)

Validación cualitativa con mensajes inventados en ambos idiomas, aplicando el threshold óptimo.

In [ ]:
def predict_message(text, pipeline=best_pipeline, threshold=0.5):
    """Devuelve etiqueta + confianza + probabilidades por clase."""
    clean = clean_text(text)
    proba = pipeline.predict_proba([clean])[0]
    classes_local = pipeline.classes_
    probs = {cls: float(p) for cls, p in zip(classes_local, proba)}
    spam_prob = probs.get("spam", 0.0)
    label = "spam" if spam_prob >= threshold else "ham"
    return {
        "text": text, "prediction": label,
        "confidence": float(max(proba)),
        "probabilities": probs,
    }


ejemplos = [
    "WIN a free iPhone now!!! Click here http://bit.ly/win-free",
    "URGENT: Your account has been suspended. Verify at http://bank-verify.cc",
    "Hey, are you coming to the meeting tomorrow at 3pm?",
    "The Q3 report is attached for your review, please send feedback by Friday.",
    "FELICIDADES! Has ganado 5000 dolares. Reclama tu premio en http://premio.cc",
    "Tu cuenta BBVA fue bloqueada por seguridad. Verifica en http://bbva-seguro.tk",
    "Hola, paso por ti a las 7 para ir al cine.",
    "Recordatorio: junta de equipo manana a las 10am en la sala 3.",
]

print(f"Usando threshold optimo = {optimal_threshold:.3f}\n")
for msg in ejemplos:
    result = predict_message(msg, threshold=optimal_threshold)
    label = result["prediction"].upper()
    icon = "[X]" if label == "SPAM" else "[OK]"
    p_spam = result["probabilities"].get("spam", 0)
    print(f"{icon} [{label:4}] P(spam)={p_spam:.2%}  {msg[:90]}")

# 20. Persistencia del modelo

Guardamos el pipeline completo + threshold óptimo + métricas en un único `.joblib`.

In [ ]:
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)
model_path = MODELS_DIR / "spam_classifier.joblib"

artifact = {
    "pipeline": best_pipeline,
    "threshold": optimal_threshold,
    "metrics": best_metrics,
    "trained_on": {"n_train": len(X_train), "n_test": len(X_test)},
}
joblib.dump(artifact, model_path)
print(f"Modelo guardado en: {model_path.resolve()}")
print(f"Tamano en disco: {model_path.stat().st_size / 1024:.1f} KB")

loaded = joblib.load(model_path)
test_msg = "Free entry in a weekly competition, text WIN to 80086"
result = predict_message(test_msg, pipeline=loaded["pipeline"], threshold=loaded["threshold"])
print(f"\nPrueba de modelo cargado:")
print(f"  Mensaje:    {test_msg}")
print(f"  Prediccion: {result['prediction'].upper()} (conf={result['confidence']:.2%})")

# 21. Conclusiones y limitaciones

## Conclusiones

1. **Pipeline bilingüe funcional.** Combinando UCI SMS (inglés) + dataset/seed español + char n-grams se obtiene detección robusta en ambos idiomas.
2. **Cross-validation confirma generalización.** Las brechas train vs CV son pequeñas (<0.05), descartando overfitting severo.
3. **GridSearchCV mejoró el modelo base** seleccionando automáticamente los mejores hiperparámetros.
4. **El tuning de threshold sube Recall** sin degradar demasiado Precision, alineando el modelo con la métrica prioritaria del proyecto.
5. **La evaluación por idioma reveló diferencias** entre EN y ES; sirve como guía para futuras mejoras (más datos en español).

## Limitaciones

1. **Desbalance de idiomas.** El español sigue sub-representado; char n-grams ayudan pero no compensan totalmente.
2. **El spam evoluciona.** Modelos entrenados hoy pueden quedar obsoletos en meses; se necesita reentrenamiento periódico.
3. **Sin contexto del remitente.** Clasificamos solo texto. En producción se combinaría con reputación del remitente, DKIM, SPF, etc.
4. **Sensibilidad a adversarios.** Un atacante que conozca el modelo puede ofuscar palabras (`v1agr@`) para evadirlo.
5. **GridSearch acotado.** Por costo computacional probamos solo 18 combinaciones; un grid mayor podría encontrar parámetros aún mejores.

## Trabajo futuro

- Probar **Transformers multilingües** (XLM-RoBERTa, mBERT) para capturar contexto semántico.
- Aumentar el dataset español con phishing real recopilado de CERT-Es / INCIBE.
- Implementar **active learning**: re-etiquetar manualmente los casos donde el modelo está menos seguro.
- Desplegar como **API REST** con FastAPI.
- Probar **modelos calibrados** (`CalibratedClassifierCV`) para obtener probabilidades más confiables.

---

## Referencias

- Russell, S. & Norvig, P. — *Artificial Intelligence: A Modern Approach*.
- Almeida, T. A., Hidalgo, J. M. G. (2011) — *Contributions to the Study of SMS Spam Filtering*.
- Metsis, V., Androutsopoulos, I., Paliouras, G. (2006) — *Spam Filtering with Naive Bayes — Which Naive Bayes?*
- scikit-learn — *Working with Text Data*, *Tuning the hyper-parameters of an estimator*.

---

**Fin del notebook — Universidad Nacional de Costa Rica · Curso de Inteligencia Artificial**